In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# --- Загрузка данных из Excel ---
excel_file = "Данные для тестового задания.xlsx"  # Указано название вашего файла

try:
    listers_df = pd.read_excel(excel_file, sheet_name="Листеры")
    ab_tests_df = pd.read_excel(excel_file, sheet_name="Данные АБ тестов")
    audience_df = pd.read_excel(excel_file, sheet_name="Данные об аудитории")
except FileNotFoundError:
    print(f"Ошибка: Файл '{excel_file}' не найден. Пожалуйста, проверьте путь к файлу.")
    exit()
except ValueError as e:
    print(f"Ошибка при чтении листов Excel: {e}. Пожалуйста, проверьте названия листов.")
    exit()

# --- Предварительная обработка ---
# Для листа "Листеры"
if 'date' in listers_df.columns:
    listers_df['date'] = pd.to_datetime(listers_df['date'])
    listers_df['month'] = listers_df['date'].dt.month
else:
    print("Предупреждение: Столбец 'date' не найден в данных 'Листеры'.")

# Для листа "Данные об аудитории"
if 'date' in audience_df.columns:
    audience_df['date'] = pd.to_datetime(audience_df['date'])
    audience_df['month'] = audience_df['date'].dt.month
else:
    print("Предупреждение: Столбец 'date' не найден в данных 'Данные об аудитории'.")

# --- Задача 1: MAU продукта в ноябре ---
if 'month' in audience_df.columns and 'user_id' in audience_df.columns:
    november_mau = audience_df[audience_df['month'] == 11]['user_id'].nunique()
    print(f"1. MAU продукта в ноябре: {november_mau}")
else:
    print("1. Отсутствуют необходимые столбцы ('month', 'user_id') в данных 'Данные об аудитории' для расчета MAU.")

# --- Задача 2: DAU продукта (средний за ноябрь) ---
if 'month' in audience_df.columns and 'date' in audience_df.columns and 'user_id' in audience_df.columns:
    november_dau_df = audience_df[audience_df['month'] == 11].groupby('date')['user_id'].nunique()
    average_november_dau = november_dau_df.mean()
    print(f"2. Средний DAU продукта за ноябрь: {average_november_dau:.2f}")
else:
    print("2. Отсутствуют необходимые столбцы ('month', 'date', 'user_id') в данных 'Данные об аудитории' для расчета DAU.")

# --- Задача 3: Retention первого дня у пользователей, пришедших 1 ноября ---
if 'date' in audience_df.columns and 'user_id' in audience_df.columns:
    nov_1_users = audience_df[audience_df['date'] == pd.to_datetime('2023-11-01')]['user_id'].unique()
    nov_2_users = audience_df[audience_df['date'] == pd.to_datetime('2023-11-02')]['user_id'].unique()
    retained_nov_1_users = len(set(nov_1_users) & set(nov_2_users))
    total_nov_1_users = len(nov_1_users)
    retention_rate_nov_1 = (retained_nov_1_users / total_nov_1_users) * 100 if total_nov_1_users > 0 else 0
    print(f"3. Retention первого дня (когорта 1 ноября): {retention_rate_nov_1:.2f}%")
else:
    print("3. Отсутствуют необходимые столбцы ('date', 'user_id') в данных 'Данные об аудитории' для расчета retention.")

# --- Задача 4: Анализ retention кривых (текстовое описание) ---
print("\n4. Анализ retention кривых (текстовое описание):")
print("Для анализа retention кривых необходимо визуальное представление графика.")
print("Как правило, более высокая кривая и более медленное снижение указывают на лучшую удерживаемость пользователей.")
print("Крутой спад в начале кривой может говорить о проблемах с onboarding или ценностью продукта для новых пользователей.")
print("Стабилизация кривой на определенном уровне показывает ядро лояльных пользователей.")
print("Сравнение retention кривых разных когорт или продуктов позволяет оценить эффективность изменений или сравнить их привлекательность.")

# --- Задача 5: Пользовательская конверсия в просмотр объявления за ноябрь ---
if 'month' in audience_df.columns and 'view_adverts' in audience_df.columns and 'user_id' in audience_df.columns:
    november_audience = audience_df[audience_df['month'] == 11]
    total_november_users = november_audience['user_id'].nunique()
    viewed_ad_users = november_audience[november_audience['view_adverts'] > 0]['user_id'].nunique()
    if total_november_users > 0:
        user_conversion = (viewed_ad_users / total_november_users) * 100
        print(f"5. Пользовательская конверсия в просмотр объявления за ноябрь: {user_conversion:.1f}%")
    else:
        print("5. Нет данных за ноябрь для расчета конверсии.")
else:
    print("5. Отсутствуют необходимые столбцы ('month', 'view_adverts', 'user_id') в данных 'Данные об аудитории'.")

# --- Задача 6: Среднее количество просмотренных объявлений на пользователя в ноябре ---
if 'month' in audience_df.columns and 'view_adverts' in audience_df.columns and 'user_id' in audience_df.columns:
    november_audience = audience_df[audience_df['month'] == 11]
    if not november_audience.empty:
        average_views = november_audience['view_adverts'].mean()
        print(f"6. Среднее количество просмотренных объявлений на пользователя в ноябре: {average_views:.2f}")
    else:
        print("6. Нет данных за ноябрь для расчета среднего количества просмотров.")
else:
    print("6. Отсутствуют необходимые столбцы ('month', 'view_adverts', 'user_id') в данных 'Данные об аудитории'.")

# --- Задача 7: NPS ---
critics = 500
supporters = 1200
neutrals = 300
total_users_survey = critics + supporters + neutrals
if total_users_survey > 0:
    percentage_supporters = (supporters / total_users_survey) * 100
    percentage_critics = (critics / total_users_survey) * 100
    nps = percentage_supporters - percentage_critics
    print(f"7. NPS: {nps:.0f}%")
else:
    print("7. Общее количество пользователей в опросе равно нулю.")

# --- Задача 8: Анализ АБ-тестов для ARPU ---
print("\n--- 8. Анализ АБ-тестов для ARPU ---")
if 'experiment_num' in ab_tests_df.columns and 'experiment_group' in ab_tests_df.columns and 'revenue' in ab_tests_df.columns and 'user_id' in ab_tests_df.columns:
    for experiment in ab_tests_df['experiment_num'].unique():
        experiment_data = ab_tests_df[ab_tests_df['experiment_num'] == experiment]
        group_control = experiment_data[experiment_data['experiment_group'] == 'control']['revenue']
        group_test = experiment_data[experiment_data['experiment_group'] == 'test']['revenue']
        users_control = experiment_data[experiment_data['experiment_group'] == 'control']['user_id'].nunique()
        users_test = experiment_data[experiment_data['experiment_group'] == 'test']['user_id'].nunique()

        if users_control > 0 and users_test > 0:
            arpu_control = group_control.sum() / users_control
            arpu_test = group_test.sum() / users_test

            # Тест Стьюдента для сравнения средних ARPU
            t_statistic, p_value = stats.ttest_ind(group_control, group_test, equal_var=False, nan_policy='omit')

            print(f"\nРезультаты АБ-теста {experiment}:")
            print(f"  Средний ARPU группы control: {arpu_control:.2f}")
            print(f"  Средний ARPU группы test: {arpu_test:.2f}")
            print(f"  p-value: {p_value:.3f}")

            alpha = 0.05
            if p_value < alpha:
                if arpu_test > arpu_control:
                    print(f"  Вывод: Группа test статистически значимо улучшила ARPU.")
                    print("  Рекомендация: Внедрить изменения группы test.")
                elif arpu_control > arpu_test:
                    print(f"  Вывод: Группа control статистически значимо улучшила ARPU.")
                    print("  Рекомендация: Сохранить текущую версию (группа control).")
                else:
                    print("  Вывод: Статистически значимых различий в ARPU между группами не обнаружено.")
                    print("  Рекомендация: Продолжить тестирование или рассмотреть другие метрики.")
            else:
                print("  Вывод: Статистически значимых различий в ARPU между группами не обнаружено.")
                print("  Рекомендация: Продолжить тестирование или рассмотреть другие метрики.")
        else:
            print(f"\nРезультаты АБ-теста {experiment}: Недостаточно данных для сравнения групп control и test.")
else:
    print("\n8. Отсутствуют необходимые столбцы ('experiment_num', 'experiment_group', 'revenue', 'user_id') в данных АБ-тестов.")

# --- Задача 9: Средний доход на пользователя по датасету с листерами ---
if 'revenue' in listers_df.columns and 'user_id' in listers_df.columns:
    total_revenue_listers = listers_df['revenue'].sum()
    total_users_listers = listers_df['user_id'].nunique()
    if total_users_listers > 0:
        average_revenue_per_user_listers = total_revenue_listers / total_users_listers
        print(f"\n9. Средний доход на пользователя по датасету с листерами: {average_revenue_per_user_listers:.2f}")
    else:
        print("9. Нет пользователей в датасете с листерами.")
else:
    print("9. Отсутствуют необходимые столбцы ('revenue', 'user_id') в данных 'Листеры'.")

# --- Задача 10: Медиана возраста пользователя по датасету с листерами ---
if 'age' in listers_df.columns:
    median_age_listers = listers_df['age'].median()
    print(f"\n10. Медиана возраста пользователя по датасету с листерами: {median_age_listers:.1f}")
else:
    print("10. Отсутствует столбец 'age' в данных 'Листеры'.")

# --- Задача 11: График для отображения разброса цен на товары в разных магазинах ---
print("\n11. Графики, которые лучше всего подходят для отображения разброса цен на товары в разных магазинах:")
print("* Ящик с усами (box plot)")
print("* Скрипичная диаграмма (violin plot)")

# --- Код для построения графика (при наличии столбцов 'store' и 'price' в listers_df) ---
if 'store' in listers_df.columns and 'price' in listers_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='store', y='price', data=listers_df)
    plt.title('Разброс цен на товары по магазинам')
    plt.xlabel('Магазин')
    plt.ylabel('Цена')
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.violinplot(x='store', y='price', data=listers_df)
    plt.title('Распределение цен на товары по магазинам')
    plt.xlabel('Магазин')
    plt.ylabel('Цена')
    plt.grid(True)
    plt.show()
else:
    print("\nПредупреждение: Отсутствуют столбцы 'store' или 'price' в данных 'Листеры' для построения графика.")

1. MAU продукта в ноябре: 7639
2. Средний DAU продукта за ноябрь: 560.47
3. Retention первого дня (когорта 1 ноября): 26.65%

4. Анализ retention кривых (текстовое описание):
Для анализа retention кривых необходимо визуальное представление графика.
Как правило, более высокая кривая и более медленное снижение указывают на лучшую удерживаемость пользователей.
Крутой спад в начале кривой может говорить о проблемах с onboarding или ценностью продукта для новых пользователей.
Стабилизация кривой на определенном уровне показывает ядро лояльных пользователей.
Сравнение retention кривых разных когорт или продуктов позволяет оценить эффективность изменений или сравнить их привлекательность.
5. Пользовательская конверсия в просмотр объявления за ноябрь: 46.3%
6. Среднее количество просмотренных объявлений на пользователя в ноябре: 1.30
7. NPS: 35%

--- 8. Анализ АБ-тестов для ARPU ---

Результаты АБ-теста 1:
  Средний ARPU группы control: 722.46
  Средний ARPU группы test: 665.74
  p-value: 0.68